# 외부 데이터 수집 파이프라인

| 스텝 | 내용 | 소스 |
|---|---|---|
| 1 | 지하철역 | t-data API / CSV |
| 2 | 학교(초·중) | NEIS API / CSV |
| 3 | 공원·하천 | data.go.kr CSV |
| 4 | 정비구역 | V-World WFS |
| 5 | 상권 | 소상공인 CSV |
| - | 단지 점수 산출 | DB 기반 |

**CSV 파일 준비:**
- `raw/park/전국도시공원정보표준데이터.csv` → [data.go.kr/15012890](https://www.data.go.kr/data/15012890/standard.do) (cp949)
- `raw/shop/소상공인시장진흥공단_상가(상권)정보.csv` → [data.go.kr/15083033](https://www.data.go.kr/data/15083033/fileData.do) (utf-8)

> API 키 만료일: **2028-04-30** (발급일 2026-04-30 기준 2년)


## 1. 환경 설정

In [ ]:
import os, csv, json, math, time, sqlite3, requests
from pathlib import Path
from datetime import datetime, date, timedelta
from collections import defaultdict
from dotenv import load_dotenv

try:
    from tqdm.notebook import tqdm
except ImportError:
    def tqdm(it, **kw): return it

# 프로젝트 루트 (노트북 위치 기준)
ROOT    = Path.cwd()
DB_PATH = ROOT / 'realestate.db'
RAW     = ROOT / 'raw'

RAW_SUBWAY = RAW / 'subway'
RAW_SCHOOL = RAW / 'school'
RAW_PARK   = RAW / 'park'
RAW_SHOP   = RAW / 'shop'

for d in [RAW_SUBWAY, RAW_SCHOOL, RAW_PARK, RAW_SHOP]:
    d.mkdir(parents=True, exist_ok=True)

print(f'ROOT:    {ROOT}')
print(f'DB:      {DB_PATH}')


## 2. API 키 로딩 및 만료일 확인

In [ ]:
# .env 또는 config/secrets.json 에서 키 로드
load_dotenv(ROOT / '.env')

_cfg = {}
_cfg_path = ROOT / 'config' / 'secrets.json'
if _cfg_path.exists():
    _cfg = json.loads(_cfg_path.read_text(encoding='utf-8'))

def get_key(name):
    return _cfg.get(name) or os.getenv(name, '')

API_KEYS = {
    'tdata':  get_key('TDATA_API_KEY'),
    'neis':   get_key('NEIS_API_KEY'),
    'vworld': get_key('VWORLD_API_KEY'),
}

API_KEY_META = {
    'tdata':  {'name': '지하철역 GEOM (t-data)',    'issued_at': date(2026,4,30), 'expires_in': 730,
               'renew_url': 'https://t-data.seoul.go.kr/userguide/guideopenapi.do'},
    'neis':   {'name': '학교알리미 (open.neis.go.kr)', 'issued_at': date(2026,4,30), 'expires_in': 730,
               'renew_url': 'https://open.neis.go.kr/portal/mypage/keyIssue.do'},
    'vworld': {'name': 'V-World WFS (vworld.kr)',   'issued_at': date(2026,4,30), 'expires_in': 730,
               'renew_url': 'https://www.vworld.kr/dev/v4dv_openapilist_s002.do'},
}

print('[API 키 상태]')
today = date.today()
for k, meta in API_KEY_META.items():
    expire = meta['issued_at'] + timedelta(days=meta['expires_in'])
    left   = (expire - today).days
    status = ('만료됨' if left <= 0 else
              f'{left}일 후 만료 ({expire})' if left <= 90 else
              f'{left}일 남음 ({expire})')
    val    = 'OK' if API_KEYS[k] else '미설정'
    print(f'  {k:8s}: [{val}]  {status}')
    if left <= 30:
        print(f'           재발급: {meta["renew_url"]}')


## 3. 공통 유틸

In [ ]:
def haversine(lat1, lng1, lat2, lng2) -> float:
    R = 6_371_000
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = math.radians(lat2-lat1), math.radians(lng2-lng1)
    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))

SUBWAY_LINE_GRADE = {
    '2호선':1,'5호선':1,'9호선':1,'9호선(연장)':1,
    '신분당선':1,'신분당선(연장)':1,'신분당선(연장2)':1,
    '수도권 광역급행철도':2,'3호선':2,'7호선':2,'분당선':2,
    '수인선':2,'경의중앙선':2,'공항철도1호선':2,
    '1호선':3,'4호선':3,'6호선':3,'8호선':3,
    '우이신설선':3,'신림선':3,'김포골드라인':3,
}
STATUS_RANK = {'관리처분인가':4,'사업시행인가':3,'조합설립인가':2,'구역지정':1,'완료':0}
GRID_STEP   = 0.005

def _grid_id(lat, lng):
    return f'{round(lat/GRID_STEP)*GRID_STEP:.4f}_{round(lng/GRID_STEP)*GRID_STEP:.4f}'

def _centroid(geom):
    try:
        gtype, coords = geom.get('type',''), geom.get('coordinates',[])
        if gtype == 'Point':       return coords[1], coords[0]
        if gtype == 'Polygon':     ring = coords[0]; return sum(p[1] for p in ring)/len(ring), sum(p[0] for p in ring)/len(ring)
        if gtype == 'MultiPolygon': pts=[p for poly in coords for ring in poly for p in ring]; return sum(p[1] for p in pts)/len(pts), sum(p[0] for p in pts)/len(pts)
    except Exception: pass
    return None, None

print('유틸 함수 로드 완료')


## 4. DB 초기화

In [ ]:
def init_db():
    con = sqlite3.connect(DB_PATH)
    con.executescript("""
    CREATE TABLE IF NOT EXISTS subway_stations (
        station_id TEXT PRIMARY KEY, station_name TEXT, line TEXT,
        line_grade INTEGER, lat REAL, lng REAL, updated_at TEXT);
    CREATE TABLE IF NOT EXISTS schools (
        school_id TEXT PRIMARY KEY, school_name TEXT, school_type TEXT,
        gu_code TEXT, lat REAL, lng REAL, updated_at TEXT);
    CREATE TABLE IF NOT EXISTS parks (
        park_id TEXT PRIMARY KEY, park_name TEXT, park_type TEXT,
        area_m2 REAL, lat REAL, lng REAL, updated_at TEXT);
    CREATE TABLE IF NOT EXISTS rivers (
        river_id TEXT PRIMARY KEY, river_name TEXT,
        lat REAL, lng REAL, updated_at TEXT);
    CREATE TABLE IF NOT EXISTS redevelopment_zones (
        zone_id TEXT PRIMARY KEY, zone_name TEXT, zone_type TEXT, status TEXT,
        gu_code TEXT, lat REAL, lng REAL, area_m2 REAL, permit_year INTEGER, updated_at TEXT);
    CREATE TABLE IF NOT EXISTS commerce_zones (
        grid_id TEXT PRIMARY KEY, lat REAL, lng REAL,
        total_stores INTEGER, restaurant_cnt INTEGER, cafe_cnt INTEGER,
        hospital_cnt INTEGER, pharmacy_cnt INTEGER, convenience_cnt INTEGER,
        commerce_score REAL, updated_at TEXT);
    CREATE TABLE IF NOT EXISTS apt_external (
        apt_seq TEXT PRIMARY KEY, nearest_station TEXT, nearest_station_dist REAL,
        nearest_line TEXT, line_grade INTEGER, walk_min REAL,
        elem_school_cnt_1km INTEGER, nearest_school_dist REAL,
        park_dist REAL, river_dist REAL, green_score REAL,
        redv_nearby INTEGER, redv_status_best TEXT,
        gtx_dist REAL, gtx_line TEXT,
        commerce_score REAL, hospital_cnt INTEGER, infra_score REAL, updated_at TEXT);
    """)
    con.commit(); con.close()
    print(f'DB 초기화 완료: {DB_PATH}')

init_db()


## Step 1. 지하철역
> CSV 우선(`raw/subway/*.csv`), 없으면 t-data API 호출

In [ ]:
def step1_subway():
    print('[1/5] 지하철역 수집')
    csv_files = list(RAW_SUBWAY.glob('*.csv'))
    if csv_files:
        print(f'  CSV: {csv_files[0].name}')
        con = sqlite3.connect(DB_PATH); saved = skipped = 0
        with open(csv_files[0], encoding='utf-8-sig') as f:
            for row in csv.DictReader(f):
                try:
                    lng, lat = float(row['환승역X좌표']), float(row['환승역Y좌표'])
                    if not lat or not lng: skipped+=1; continue
                    line  = row['호선명칭'].strip()
                    grade = SUBWAY_LINE_GRADE.get(line, 3)
                    con.execute('INSERT OR REPLACE INTO subway_stations VALUES (?,?,?,?,?,?,?)',
                                (row['외구간_역_수'], row['역한글명칭'], line, grade, lat, lng, datetime.now().isoformat()))
                    saved += 1
                except Exception: skipped += 1
        con.commit(); con.close()
        print(f'  저장: {saved:,}개 (스킵 {skipped})')
        return

    key = API_KEYS['tdata']
    if not key:
        print('  TDATA_API_KEY 미설정 — raw/subway/ 에 CSV 파일을 넣거나 .env에 키 추가')
        print('  다운로드: https://t-data.seoul.go.kr')
        return
    BASE = 'http://t-data.seoul.go.kr/apig/apiman-gateway/tapi/TaimsKsccDvSubwayStationGeom/1.0'
    rows, start = [], 1
    while True:
        res = requests.get(BASE, params={'apikey':key,'startRow':start,'rowCnt':100}, timeout=10).json()
        batch = res if isinstance(res, list) else res.get('result',[])
        if not batch: break
        rows.extend(batch); start += 100; time.sleep(0.2)
        if len(batch) < 100: break
    con = sqlite3.connect(DB_PATH); saved = 0
    for r in rows:
        try:
            lat, lng = float(r.get('convY',0)), float(r.get('convX',0))
            if not lat or not lng: continue
            line  = r.get('lineNm','').strip()
            grade = SUBWAY_LINE_GRADE.get(line, 3)
            con.execute('INSERT OR REPLACE INTO subway_stations VALUES (?,?,?,?,?,?,?)',
                        (str(r.get('outStnNum','')), r.get('stnKrNm',''), line, grade, lat, lng, datetime.now().isoformat()))
            saved += 1
        except Exception: continue
    con.commit(); con.close()
    print(f'  저장: {saved:,}개')

step1_subway()


## Step 2. 학교 (초·중)
> CSV 우선(`raw/school/*.csv`), 없으면 NEIS API 호출

In [ ]:
SCHOOL_KIND_MAP = {'02':'elementary','03':'middle'}
EDU_CODES       = {'서울':'B10','경기':'C10','인천':'E10'}

def step2_school():
    print('[2/5] 학교 정보 수집')
    csv_files = list(RAW_SCHOOL.glob('*.csv'))
    if csv_files:
        for f in csv_files:
            con = sqlite3.connect(DB_PATH); saved = skipped = 0
            try: fh = open(f, encoding='utf-8-sig')
            except UnicodeDecodeError: fh = open(f, encoding='cp949')
            for row in csv.DictReader(fh):
                try:
                    lat = float(row.get('LTTUD') or row.get('위도') or 0)
                    lng = float(row.get('LGTUD') or row.get('경도') or 0)
                    if not lat or not lng or row.get('ABSCH_YN')=='Y': skipped+=1; continue
                    stype = SCHOOL_KIND_MAP.get(row.get('SCHUL_KND_SC_CODE','').strip(),'other')
                    con.execute('INSERT OR REPLACE INTO schools VALUES (?,?,?,?,?,?,?)',
                                (row.get('SCHUL_CODE',''), row.get('SCHUL_NM',''), stype,
                                 str(row.get('ADRCD_CD',''))[:5], lat, lng, datetime.now().isoformat()))
                    saved += 1
                except Exception: skipped += 1
            fh.close(); con.commit(); con.close()
            print(f'  {f.name}: {saved:,}개 저장')
        return

    key = API_KEYS['neis']
    if not key:
        print('  NEIS_API_KEY 미설정 — .env에 키 추가')
        print('  발급: https://open.neis.go.kr/portal/mypage/keyIssue.do')
        return
    BASE = 'https://open.neis.go.kr/hub/schoolInfo'
    for region, edu_code in EDU_CODES.items():
        for kind_code, label in [('02','초등'),('03','중학')]:
            print(f'  {region} {label}...')
            page, rows = 1, []
            while True:
                data  = requests.get(BASE, params={'KEY':key,'Type':'json','pIndex':page,'pSize':1000,
                                                   'ATPT_OFCDC_ORG_CODE':edu_code,'SCHUL_KND_SC_CODE':kind_code}, timeout=15).json()
                info  = data.get('schoolInfo',[])
                if len(info) < 2: break
                batch = info[1].get('row',[])
                if not batch: break
                rows.extend(batch); page += 1; time.sleep(0.2)
                if len(batch) < 1000: break
            con = sqlite3.connect(DB_PATH); saved = 0
            for it in rows:
                try:
                    if it.get('ABSCH_YN')=='Y': continue
                    lat, lng = float(it.get('LTTUD',0) or 0), float(it.get('LGTUD',0) or 0)
                    if not lat or not lng: continue
                    stype = SCHOOL_KIND_MAP.get(it.get('SCHUL_KND_SC_CODE','').strip(),'other')
                    con.execute('INSERT OR REPLACE INTO schools VALUES (?,?,?,?,?,?,?)',
                                (it.get('SCHUL_CODE',''), it.get('SCHUL_NM',''), stype,
                                 str(it.get('ADRCD_CD',''))[:5], lat, lng, datetime.now().isoformat()))
                    saved += 1
                except Exception: continue
            con.commit(); con.close()
            print(f'    {saved:,}개 저장')

step2_school()


## Step 3. 공원·하천
> `raw/park/전국도시공원정보표준데이터.csv` (cp949) 필요

In [ ]:
RIVERS = [
    ('R001','한강',37.5326,126.9903),('R002','청계천',37.5705,126.9942),
    ('R003','안양천',37.5065,126.8880),('R004','중랑천',37.5721,127.0721),
    ('R005','홍제천',37.5833,126.9426),('R006','불광천',37.6101,126.9175),
    ('R007','탄천',37.4767,127.1208),('R008','도림천',37.4960,126.9025),
    ('R009','우이천',37.6362,127.0290),('R010','경안천',37.4001,127.2561),
    ('R011','황구지천',37.2980,127.0370),('R012','수원천',37.2856,127.0166),
    ('R013','굴포천',37.5132,126.7490),('R014','왕숙천',37.6490,127.2190),
    ('R015','신천',37.3559,127.1130),
]

def step3_park():
    print('[3/5] 공원·하천 수집')
    csv_files = list(RAW_PARK.glob('*.csv'))
    con = sqlite3.connect(DB_PATH)
    for rid, rname, rlat, rlng in RIVERS:
        con.execute('INSERT OR REPLACE INTO rivers VALUES (?,?,?,?,?)',
                    (rid, rname, rlat, rlng, datetime.now().isoformat()))
    con.commit()
    print(f'  하천 {len(RIVERS)}개 저장')

    if not csv_files:
        print('  raw/park/ 에 CSV 없음 — 하천만 저장됨')
        print('  다운로드: https://www.data.go.kr/data/15012890/standard.do')
        con.close(); return

    saved = skipped = 0
    with open(csv_files[0], encoding='cp949') as f:
        for row in csv.DictReader(f):
            try:
                lat, lng = float(row.get('위도',0) or 0), float(row.get('경도',0) or 0)
                if not (32<lat<39 and 124<lng<132): skipped+=1; continue
                pid = row.get('관리번호','').strip()
                if not pid: skipped+=1; continue
                con.execute('INSERT OR REPLACE INTO parks VALUES (?,?,?,?,?,?,?)',
                            (pid, row.get('공원명',''), row.get('공원구분','기타'),
                             float(row.get('공원면적',0) or 0), lat, lng, datetime.now().isoformat()))
                saved += 1
            except Exception: skipped += 1
    con.commit(); con.close()
    print(f'  공원 {saved:,}개 저장 (스킵 {skipped:,})')

step3_park()


## Step 4. 정비구역 (V-World WFS)

In [ ]:
REDV_TYPE_MAP = {'UDR':'재개발구역','UDS':'주거환경개선지구','UDT':'정비구역','UDQ':'토지구획정리'}
SUDOGWON_BBOX = '126.60,37.10,127.70,37.90'

def step4_redevelopment():
    print('[4/5] 정비구역 수집 (V-World WFS)')
    key = API_KEYS['vworld']
    if not key:
        print('  VWORLD_API_KEY 미설정 — .env에 키 추가')
        print('  발급: https://www.vworld.kr')
        return
    BASE = 'https://api.vworld.kr/req/wfs'
    CQL  = "UGRND_USE_SYM LIKE 'UDR%' OR UGRND_USE_SYM LIKE 'UDS%' OR UGRND_USE_SYM LIKE 'UDT%' OR UGRND_USE_SYM LIKE 'UDQ%'"
    features, start = [], 0
    while True:
        try:
            data  = requests.get(BASE, params={'SERVICE':'WFS','REQUEST':'GetFeature','VERSION':'2.0.0',
                                               'TYPENAME':'lp_pa_rdscd_indtp','SRSNAME':'EPSG:4326',
                                               'OUTPUT':'application/json',
                                               'BBOX':SUDOGWON_BBOX+',EPSG:4326','CQL_FILTER':CQL,
                                               'COUNT':1000,'STARTINDEX':start,'KEY':key}, timeout=30).json()
            batch = data.get('features',[])
            if not batch: break
            features.extend(batch)
            print(f'  {len(features):,}건 수집 중...')
            if len(batch) < 1000: break
            start += 1000; time.sleep(0.3)
        except Exception as e: print(f'  WFS 오류: {e}'); break

    con = sqlite3.connect(DB_PATH); saved = skipped = 0
    for feat in features:
        try:
            props = feat.get('properties',{})
            lat, lng = _centroid(feat.get('geometry',{}))
            if not lat: skipped+=1; continue
            use_sym = props.get('UGRND_USE_SYM','')
            ztype   = next((v for k,v in REDV_TYPE_MAP.items() if use_sym.startswith(k)),'기타')
            zone_id = props.get('PNU', feat.get('id', f'RD_{saved}'))
            con.execute('INSERT OR REPLACE INTO redevelopment_zones VALUES (?,?,?,?,?,?,?,?,?,?)',
                        (zone_id, props.get('UGRND_USE_NM',ztype), ztype, '구역지정',
                         str(props.get('PNU',''))[:5], lat, lng, 0.0, 0, datetime.now().isoformat()))
            saved += 1
        except Exception: skipped += 1
    con.commit(); con.close()
    print(f'  저장: {saved:,}개 (스킵 {skipped:,})')

step4_redevelopment()


## Step 5. 상권
> `raw/shop/소상공인시장진흥공단_상가(상권)정보.csv` (utf-8, 수백MB)

In [ ]:
COMMERCE_KEYWORDS = {
    'restaurant':  ['한식','중식','일식','양식','분식','치킨','피자','고기','찜','탕'],
    'cafe':        ['커피','카페','베이커리','제과','디저트','아이스크림'],
    'hospital':    ['병원','의원','한의원','치과','내과','외과','소아과','피부과'],
    'pharmacy':    ['약국'],
    'convenience': ['편의점'],
}

def step5_commerce():
    print('[5/5] 상권 수집')
    csv_files = list(RAW_SHOP.glob('*.csv'))
    if not csv_files:
        print('  raw/shop/ 에 CSV 없음')
        print('  다운로드: https://www.data.go.kr/data/15083033/fileData.do')
        return
    csv_path  = csv_files[0]
    file_size = csv_path.stat().st_size / 1024**2
    print(f'  {csv_path.name} ({file_size:.0f}MB) 처리 중...')

    grids = defaultdict(lambda: {'total':0,'restaurant':0,'cafe':0,
                                  'hospital':0,'pharmacy':0,'convenience':0,'lat':0.,'lng':0.})
    processed = filtered = 0
    with open(csv_path, encoding='utf-8') as f:
        reader  = csv.DictReader(f)
        headers = reader.fieldnames or []
        lat_col = next((c for c in headers if '위도' in c), '위도')
        lng_col = next((c for c in headers if '경도' in c), '경도')
        cat_col = next((c for c in headers if '소분류명' in c), '상권업종소분류명')
        for row in reader:
            processed += 1
            if processed % 500_000 == 0:
                print(f'  {processed:,}건 처리 중 (격자 {len(grids):,}개)...')
            try:
                lat, lng = float(row.get(lat_col,0) or 0), float(row.get(lng_col,0) or 0)
                if not (36.8<lat<38.1 and 126.5<lng<127.8): continue
                gid = _grid_id(lat, lng)
                g   = grids[gid]; g['total']+=1
                g['lat'] = round(lat/GRID_STEP)*GRID_STEP
                g['lng'] = round(lng/GRID_STEP)*GRID_STEP
                cat = row.get(cat_col,'')
                for key, kws in COMMERCE_KEYWORDS.items():
                    if any(k in cat for k in kws): g[key]+=1; break
                filtered += 1
            except Exception: continue
    print(f'  {processed:,}건 -> 수도권 {filtered:,}건 -> {len(grids):,}개 격자')

    con = sqlite3.connect(DB_PATH)
    for gid, g in tqdm(grids.items(), desc='DB 적재'):
        score = min(100, g['restaurant']*0.5+g['cafe']*0.8+g['hospital']*2.0+g['pharmacy']*1.5+g['convenience']*1.0)
        con.execute('INSERT OR REPLACE INTO commerce_zones VALUES (?,?,?,?,?,?,?,?,?,?,?)',
                    (gid, g['lat'], g['lng'], g['total'], g['restaurant'], g['cafe'],
                     g['hospital'], g['pharmacy'], g['convenience'], round(score,2), datetime.now().isoformat()))
    con.commit(); con.close()
    print(f'  저장: {len(grids):,}개 격자')

step5_commerce()


## 단지별 외부 점수 산출
> `apt_master` 테이블이 실거래 파이프라인에서 생성된 이후 실행

In [ ]:
def compute_scores():
    print('단지별 외부 점수 산출')
    con = sqlite3.connect(DB_PATH)
    if not con.execute("SELECT name FROM sqlite_master WHERE type='table' AND name='apt_master'").fetchone():
        print('  apt_master 없음 — 실거래 파이프라인 먼저 실행 필요')
        con.close(); return
    apts     = con.execute('SELECT apt_seq, apt_name, lat, lng FROM apt_master WHERE lat IS NOT NULL AND lng IS NOT NULL').fetchall()
    stations = con.execute('SELECT station_name, line, line_grade, lat, lng FROM subway_stations').fetchall()
    schools  = con.execute("SELECT lat, lng FROM schools WHERE school_type='elementary'").fetchall()
    parks    = con.execute('SELECT lat, lng, area_m2 FROM parks').fetchall()
    rivers   = con.execute('SELECT lat, lng FROM rivers').fetchall()
    redvs    = con.execute('SELECT lat, lng, status FROM redevelopment_zones').fetchall()
    commerce = con.execute('SELECT lat, lng, commerce_score, hospital_cnt FROM commerce_zones').fetchall()
    print(f'  단지:{len(apts):,} 역:{len(stations):,} 학교:{len(schools):,} 공원:{len(parks):,} 상권:{len(commerce):,}')

    saved = 0
    for apt_seq, apt_name, alat, alng in tqdm(apts, desc='점수 산출'):
        try:
            st       = sorted([(haversine(alat,alng,s[3],s[4]),s) for s in stations])
            st_dist  = st[0][0] if st else 99999
            st_name  = st[0][1][0] if st else ''
            st_line  = st[0][1][1] if st else ''
            st_grade = st[0][1][2] if st else 3
            elem_1km = sum(1 for s in schools if haversine(alat,alng,s[0],s[1]) <= 1000)
            sc_dists = sorted([haversine(alat,alng,s[0],s[1]) for s in schools])
            pk_dist  = min((haversine(alat,alng,p[0],p[1]) for p in parks),  default=99999)
            rv_dist  = min((haversine(alat,alng,r[0],r[1]) for r in rivers), default=99999)
            green_sc = round(max(0,50-pk_dist/100) + max(0,50-rv_dist/200), 1)
            redv_500 = [(haversine(alat,alng,r[0],r[1]),r[2]) for r in redvs if haversine(alat,alng,r[0],r[1])<=500]
            cm       = sorted([(haversine(alat,alng,c[0],c[1]),c) for c in commerce])
            cm_score = cm[0][1][2] if cm else 0
            hosp_cnt = cm[0][1][3] if cm else 0
            gw       = {1:1.0, 2:0.8, 3:0.6}.get(st_grade, 0.6)
            infra    = round(min(100, max(0,
                max(0,35-st_dist/100)*gw + min(25,elem_1km*8) +
                min(15,green_sc*0.15)    + min(15,cm_score*0.15) +
                max(0,10-len(redv_500)*(-2))
            )), 1)
            con.execute('INSERT OR REPLACE INTO apt_external VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)',
                        (apt_seq, st_name, round(st_dist), st_line, st_grade, round(st_dist/67,1),
                         elem_1km, round(sc_dists[0] if sc_dists else 99999),
                         round(pk_dist), round(rv_dist), green_sc,
                         len(redv_500), max(redv_500,key=lambda x:STATUS_RANK.get(x[1],0))[1] if redv_500 else '',
                         0.0, '', round(cm_score,1), hosp_cnt, infra, datetime.now().isoformat()))
            saved += 1
        except Exception: continue
    con.commit(); con.close()
    print(f'  저장: {saved:,}개 단지')

compute_scores()


## 결과 요약

In [ ]:
con = sqlite3.connect(DB_PATH)
tables = [
    ('subway_stations',    '지하철역'),
    ('schools',            '학교'),
    ('parks',              '공원'),
    ('rivers',             '하천'),
    ('redevelopment_zones','정비구역'),
    ('commerce_zones',     '상권 격자'),
    ('apt_external',       '단지 외부점수'),
]
print('DB 현황')
for tbl, label in tables:
    try:    cnt = con.execute(f'SELECT COUNT(*) FROM {tbl}').fetchone()[0]
    except: cnt = 0
    print(f'  {label:12s}: {cnt:>8,}건')

try:
    rows = con.execute("""
        SELECT m.apt_name, e.nearest_station, e.walk_min, e.elem_school_cnt_1km, e.infra_score
        FROM apt_external e JOIN apt_master m ON e.apt_seq = m.apt_seq
        ORDER BY e.infra_score DESC LIMIT 5
    """).fetchall()
    if rows:
        print('\n인프라 점수 TOP5')
        for r in rows:
            print(f'  {r[0]:20s} | {r[1]}역 도보{r[2]}분 | 초교{r[3]}개 | {r[4]}점')
except Exception: pass
con.close()
